# Credit Card Fraud Detection and Analysis
**IEEE-CIS Dataset | Vesta Corporation | 590,540 Transactions**  
**Academic Year 2025-26 | Shambhavi Shailendra Kshirsagar | Roll No: 143**

## Step 1: Import Libraries and Load Dataset

In [ ]:
import pandas as pd
import numpy as np
import psycopg2
import json
import os

# Load the IEEE-CIS Fraud Detection dataset
trans = pd.read_csv('train_transaction.csv')
ident = pd.read_csv('train_identity.csv')

print('Transaction shape:', trans.shape)
print('Identity shape:', ident.shape)

In [ ]:
print('Transaction columns:', trans.columns.tolist()[:12], '...')
print('\nIdentity columns:', ident.columns.tolist())
print('\nFraud distribution:\n', trans['isFraud'].value_counts())
print('\nTransaction nulls (top 20):\n', trans.isnull().sum().sort_values(ascending=False).head(20))
print('\nIdentity nulls (top 20):\n', ident.isnull().sum().sort_values(ascending=False).head(20))

## Step 2: Data Cleaning

In [ ]:
# Drop columns missing more than 50%
thresh = len(trans) * 0.5
trans_clean = trans.dropna(axis=1, thresh=thresh)

thresh2 = len(ident) * 0.5
ident_clean = ident.dropna(axis=1, thresh=thresh2)

print('Transaction columns after drop:', trans_clean.shape[1])
print('Identity columns after drop:', ident_clean.shape[1])

In [ ]:
# Keep only business-explainable transaction columns
keep_trans = [
    'TransactionID', 'isFraud', 'TransactionDT', 'TransactionAmt',
    'ProductCD', 'card1', 'card2', 'card3', 'card4', 'card5', 'card6',
    'addr1', 'addr2', 'P_emaildomain',
    'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13', 'C14',
    'D1', 'D2', 'D3', 'D4', 'D10', 'D11', 'D15',
    'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9'
]

# Only keep columns that actually exist after the drop
keep_trans = [c for c in keep_trans if c in trans_clean.columns]
trans_clean = trans_clean[keep_trans]

print('Final transaction columns:', trans_clean.shape[1])
print('Columns kept:', trans_clean.columns.tolist())

## Step 3: Feature Engineering

In [ ]:
# Convert transaction time to hour of day and day of week
# TransactionDT is seconds offset from a reference point
trans_clean['hour'] = (trans_clean['TransactionDT'] // 3600) % 24
trans_clean['day_of_week'] = (trans_clean['TransactionDT'] // (3600 * 24)) % 7

# Amount buckets for dashboard segmentation
trans_clean['amount_bucket'] = pd.cut(
    trans_clean['TransactionAmt'],
    bins=[0, 50, 200, 500, 1000, 99999],
    labels=['0-50', '50-200', '200-500', '500-1000', '1000+']
)

# Flag high value transactions
trans_clean['is_high_value'] = (trans_clean['TransactionAmt'] > 500).astype(int)

# Email domain match flag
if 'P_emaildomain' in trans_clean.columns and 'R_emaildomain' in trans_clean.columns:
    trans_clean['email_match'] = (
        trans_clean['P_emaildomain'] == trans_clean['R_emaildomain']
    ).astype(int)
else:
    trans_clean['email_match'] = 0

print('Feature engineering complete.')
print('New columns:', ['hour', 'day_of_week', 'amount_bucket', 'is_high_value', 'email_match'])
print('\nAmount bucket distribution:')
print(trans_clean['amount_bucket'].value_counts())
print('\nFraud by hour (top 5):')
print(trans_clean.groupby('hour')['isFraud'].sum().sort_values(ascending=False).head())

## Step 4: Clean Identity Table and Merge

In [ ]:
keep_ident = [
    'TransactionID', 'DeviceType', 'DeviceInfo',
    'id_12', 'id_13', 'id_15', 'id_16', 'id_17',
    'id_19', 'id_20', 'id_28', 'id_29', 'id_31'
]
keep_ident = [c for c in keep_ident if c in ident_clean.columns]
ident_clean = ident_clean[keep_ident]

# Merge
df = trans_clean.merge(ident_clean, on='TransactionID', how='left')

print('Final merged shape:', df.shape)
print('\nFraud distribution:\n', df['isFraud'].value_counts())
print('\nRemaining nulls (top 15):\n', df.isnull().sum().sort_values(ascending=False).head(15))

## Step 5: Fill Nulls

In [ ]:
# Categorical columns fill with 'Unknown'
cat_cols = ['ProductCD', 'card4', 'card6', 'P_emaildomain',
            'M1','M2','M3','M4','M5','M6','M7','M8','M9',
            'DeviceType', 'DeviceInfo',
            'id_12', 'id_15', 'id_16', 'id_17',
            'id_19', 'id_20', 'id_28', 'id_29', 'id_31']

for col in cat_cols:
    if col in df.columns:
        df[col] = df[col].fillna('Unknown')

# Numeric columns fill with median
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

# amount_bucket fill
df['amount_bucket'] = df['amount_bucket'].astype(str).fillna('Unknown')

print('Nulls remaining:', df.isnull().sum().sum())

## Step 6: Split into Relational Tables and Export

In [ ]:
# Helper: only keep columns that exist
def safe_select(df, cols):
    return df[[c for c in cols if c in df.columns]].copy()

# Table 1: Core fact table
transactions = safe_select(df, [
    'TransactionID', 'TransactionDT', 'TransactionAmt', 'ProductCD',
    'isFraud', 'hour', 'day_of_week', 'amount_bucket',
    'is_high_value', 'email_match', 'P_emaildomain'
])

# Table 2: Card info
card_info = safe_select(df, [
    'TransactionID', 'card1', 'card2', 'card3', 'card4', 'card5', 'card6'
])

# Table 3: Address info
address_info = safe_select(df, ['TransactionID', 'addr1', 'addr2'])

# Table 4: Identity info
identity_info = safe_select(df, [
    'TransactionID', 'DeviceType', 'DeviceInfo',
    'id_12', 'id_13', 'id_15', 'id_16', 'id_17',
    'id_19', 'id_20', 'id_28', 'id_29', 'id_31'
])

# Table 5: Behavior signals
behavior = safe_select(df, [
    'TransactionID',
    'C1','C2','C3','C4','C5','C6','C7','C8','C9','C10','C11','C12','C13','C14',
    'D1','D2','D3','D4','D10','D11','D15'
])

# Export all 5 tables
os.makedirs('C:/temp', exist_ok=True)
transactions.to_csv('C:/temp/transactions.csv', index=False)
card_info.to_csv('C:/temp/card_info.csv', index=False)
address_info.to_csv('C:/temp/address_info.csv', index=False)
identity_info.to_csv('C:/temp/identity_info.csv', index=False)
behavior.to_csv('C:/temp/behavior.csv', index=False)

print('All 5 tables exported')
print(f'  transactions : {len(transactions):,} rows')
print(f'  card_info    : {len(card_info):,} rows')
print(f'  address_info : {len(address_info):,} rows')
print(f'  identity_info: {len(identity_info):,} rows')
print(f'  behavior     : {len(behavior):,} rows')

## Step 7: Export PostgreSQL Views to CSV

In [ ]:
conn = psycopg2.connect(
    host='127.0.0.1',
    database='fraud_detection',
    user='postgres',
    password='admin123'
)

views = [
    'vw_fraud_overview',
    'vw_fraud_by_product',
    'vw_fraud_by_hour',
    'vw_fraud_by_amount',
    'vw_fraud_by_device',
    'vw_fraud_by_card',
    'vw_fraud_by_day'
]

for view in views:
    df_view = pd.read_sql(f'SELECT * FROM {view}', conn)
    df_view.to_csv(f'C:/temp/{view}.csv', index=False)
    print(f'Exported {view}: {len(df_view)} rows')

conn.close()
print('\nAll views exported successfully')

## Step 8: Export Fraud Alerts and Dashboard Data

In [ ]:
conn = psycopg2.connect(
    host='127.0.0.1',
    database='fraud_detection',
    user='postgres',
    password='admin123'
)

# Fraud alerts with risk levels
query = """
SELECT
    t.TransactionID,
    t.TransactionAmt,
    t.ProductCD,
    t.hour,
    t.P_emaildomain,
    t.isFraud,
    c.card4 as card_network,
    c.card6 as card_type,
    i.DeviceType,
    CASE
        WHEN t.TransactionAmt > 1000 AND t.isFraud = 1 THEN 'CRITICAL'
        WHEN t.TransactionAmt > 500  AND t.isFraud = 1 THEN 'HIGH'
        WHEN t.TransactionAmt > 200  AND t.isFraud = 1 THEN 'MEDIUM'
        WHEN t.isFraud = 1 THEN 'LOW'
        ELSE 'NORMAL'
    END as risk_level
FROM transactions t
LEFT JOIN card_info c ON t.TransactionID = c.TransactionID
LEFT JOIN identity_info i ON t.TransactionID = i.TransactionID
WHERE t.isFraud = 1
ORDER BY t.TransactionAmt DESC
"""

alerts = pd.read_sql(query, conn)
alerts.to_csv('C:/temp/fraud_alerts.csv', index=False)
print(f'Fraud alerts exported: {len(alerts)} rows')
print('\nRisk level distribution:')
print(alerts['risk_level'].value_counts())

# Avg comparison
avg_query = """
SELECT 'Fraud' as transaction_type,
    ROUND(AVG(TransactionAmt)::numeric, 2) as avg_amount
FROM transactions WHERE isFraud = 1
UNION ALL
SELECT 'Legitimate' as transaction_type,
    ROUND(AVG(TransactionAmt)::numeric, 2) as avg_amount
FROM transactions WHERE isFraud = 0
"""
avg_df = pd.read_sql(avg_query, conn)
avg_df.to_csv('C:/temp/vw_avg_comparison.csv', index=False)
print('\nAvg comparison:')
print(avg_df)

conn.close()
print('\nAll data exported successfully')

## Key Findings Summary

In [ ]:
print('=== KEY FINDINGS ===')
print(f'Total Transactions : {len(df):,}')
print(f'Total Fraud Cases  : {df["isFraud"].sum():,}')
print(f'Fraud Rate         : {df["isFraud"].mean()*100:.2f}%')
print(f'Avg Fraud Amount   : ${df[df["isFraud"]==1]["TransactionAmt"].mean():.2f}')
print(f'Avg Legit Amount   : ${df[df["isFraud"]==0]["TransactionAmt"].mean():.2f}')
print()
print('Fraud Rate by Product Category:')
print(df.groupby('ProductCD').apply(
    lambda x: pd.Series({
        'fraud_count': x['isFraud'].sum(),
        'fraud_rate_%': round(x['isFraud'].mean()*100, 2)
    })
).sort_values('fraud_rate_%', ascending=False))
print()
print('Fraud by Device Type:')
print(df.groupby('DeviceType')['isFraud'].agg(['sum','mean']).rename(
    columns={'sum':'fraud_count','mean':'fraud_rate'}))